# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

**Выполнение:** запустите ячейки сверху вниз. Для повторного запуска в Google Colab загрузите `orders.csv` из LMS при появлении диалога. CSV в репозиторий сдавать не нужно.


## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения (до запуска проверочной ячейки):

1. `[1. 2. 3.] [0 1 1]` - обычное `/` выполняет деление с дробным результатом, `//` — целочисленное деление с округлением вниз.
2. `[4. 4. 4.]` — `m` имеет форму `(3, 4)` и состоит из единиц; `axis=1` складывает четыре числа отдельно в каждой из трех строк.
3. Матрица 3 × 3: `[[0, 1, 2], [1, 2, 3], [2, 3, 4]]` - broadcasting складывает столбец формы `(3, 1)` со строкой формы `(3,)`.
4. `6.0` - корни из `[1, 4, 9]` равны `[1., 2., 3.]`, их сумма равна 6.
5. `7.5` - квадраты чисел `[1, 2, 3, 4]` равны `[1, 4, 9, 16]`, среднее равно 30/4.


In [2]:
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [3]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [4]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [5]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [21]:
september = amount[clean & (month == '2026-09')].sum()
october = amount[clean & (month == '2026-10')].sum()
print('Сентябрь:', september)
print('Октябрь:', october)
print('Изменение в %:', (october / september - 1) * 100)

Сентябрь: 35243410.0
Октябрь: 34419520.0
Изменение в %: -2.337713632137184


после очистки выручка в октябре немного снизилась - совпадает с выводом дз 2

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [20]:
months = np.array(['2026-09', '2026-10'])
categories = np.unique(cat)
category_table = np.array([[amount[clean & (cat == c) & (month == m)].sum()
                            for m in months] for c in categories])
print('Категории:', categories)
print('Сентябрь / Октябрь):')
print(category_table)
totals = category_table.sum(axis=1)
changes = (category_table[:, 1] / category_table[:, 0] - 1) * 100
print('Больше всего выручки за 2 мес:', categories[totals.argmax()])
print('Сильнее всех упала в %:', categories[changes.argmin()])

Категории: ['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
Сентябрь / Октябрь):
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Больше всего выручки за 2 мес: Электроника
Сильнее всех упала в %: Одежда


больше всего выручки за 2 мес приносит Электроника, а сильнее всех упала одежда

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [10]:
counts = np.array([[(clean & (cat == c) & (month == m)).sum()
                    for m in months]
                    for c in categories])
average_checks = category_table / counts
print('Средний чек по категориям и месяцам:')
print(average_checks)
print('Самый высокий средний чек у категории:', categories[average_checks[:, 1].argmax()])

Средний чек по категориям и месяцам:
[[10499.43277311 10768.46638655]
 [ 4020.          3815.75250836]
 [ 6187.08530806  6158.59770115]
 [13520.79646018 13045.59815951]
 [32772.74952919 33600.1980198 ]]
Самый высокий средний чек у категории: Электроника


Самый высокий средний чек у категории Электроника в октябре = 33600

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [18]:
cities = np.unique(city)
table = np.array([[amount[clean & (city == c) & (month == m)].sum()
                   for m in months] for c in cities])
share = table / table.sum(axis=0)
print('Города:', cities)
print('Доли по городам (сентябрь / октябрь):')
print(share)
print('Сумма долей каждого месяця:', share.sum(axis=0))

for c, s in zip(cities, share):
    if c == 'Шымкент':
        print('Шымкент:', s * 100, '%')
        print('Доля Шымкента изменилась на:', (s[1] - s[0]) * 100)

Города: ['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
Доли по городам (сентябрь / октябрь):
[[0.07311892 0.07380754]
 [0.41145479 0.42542778]
 [0.25633785 0.25713694]
 [0.02621738 0.02817762]
 [0.10352347 0.09795837]
 [0.12934759 0.11749176]]
Сумма долей каждого месяця: [1. 1.]
Шымкент: [12.93475858 11.74917605] %
Доля Шымкента изменилась на: -1.1855825342961728


Доля Шымкента снизилась

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [14]:
city_change = table[:, 1] - table[:, 0]
total_change = city_change.sum()
contribution = city_change / total_change * 100
print('Изменение выручки по городам:', city_change)
print('Общее изменение:', total_change)
for c, change, percent in zip(cities, city_change, contribution):
    print(c, ':', change, ', % падения:', percent)

Изменение выручки по городам: [ -36540.  141950. -183690.   45870. -276840. -514640.]
Общее изменение: -823890.0
Актобе : -36540.0 , % падения: 4.435058078141499
Алматы : 141950.0 , % падения: -17.22924176771171
Астана : -183690.0 , % падения: 22.29545206277537
Атырау : 45870.0 , % падения: -5.567490805811456
Караганда : -276840.0 , % падения: 33.60157302552525
Шымкент : -514640.0 , % падения: 62.46464940708104


необходимо проверить снижение продаж в Шымкенте

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [16]:
print(table.shape)
print(table.sum(axis=1).shape)
within_city_share = table / table.sum(axis=1, keepdims=True)
print('Доли двух месяцев в каждом из городов:')
print(within_city_share)

(6, 2)
(6,)
Доли двух месяцев в каждом из городов:
[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]


тк table имеет форму: (6, 2), а table.sum(axis=1): (6,). Размеры просто 2 и 6 несовместимы.
видно соотношение сентября и октября для каждого города

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8